In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df

lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_part(data_folder: str, **storage_options) -> pd.DataFrame:
    data_path = f"{data_folder}/part.tbl"
    # use the C engine and memory-map the file for lower memory footprint and faster parsing
    return pd.read_csv(
        data_path,
        sep="|",
        engine="c",
        memory_map=True,
    )

part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(data_folder: str, **storage_options) -> pd.DataFrame:
    return pd.read_csv(
        f"{data_folder}/nation.tbl",
        sep="|",
        **storage_options
    )

nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")  
    return df

partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

date1 = pd.Timestamp("1996-01-01")
date2 = pd.Timestamp("1997-01-01")

# Precompute nation keys for 'JORDAN'
nkeys = nation.loc[nation.N_NAME == "JORDAN", "N_NATIONKEY"].unique()

# Filter partsupp by parts whose names start with 'azure'
tmp_ps = partsupp.loc[
    partsupp.PS_PARTKEY.isin(
        part.loc[part.P_NAME.str.startswith("azure"), "P_PARTKEY"]
    ),
    ["PS_PARTKEY", "PS_SUPPKEY", "PS_AVAILQTY"]
]

# Filter lineitem by ship date and select only needed columns
li = lineitem.loc[
    (lineitem.L_SHIPDATE >= date1) & (lineitem.L_SHIPDATE < date2),
    ["L_PARTKEY", "L_SUPPKEY", "L_QUANTITY"]
]

# Join partsupp with filtered lineitem, aggregate and filter by availability
agg = (
    tmp_ps
    .merge(
        li,
        left_on=["PS_PARTKEY", "PS_SUPPKEY"],
        right_on=["L_PARTKEY", "L_SUPPKEY"]
    )
    .groupby(["PS_PARTKEY", "PS_SUPPKEY", "PS_AVAILQTY"], as_index=False, sort=False)["L_QUANTITY"].sum()
    .loc[lambda df: df.PS_AVAILQTY > 0.5 * df.L_QUANTITY]
)

# Filter supplier by Jordan nation and join to get names and addresses
tmp_sup = supplier.loc[
    supplier.S_NATIONKEY.isin(nkeys),
    ["S_SUPPKEY", "S_NAME", "S_ADDRESS"]
]

total = (
    agg
    .merge(
        tmp_sup,
        left_on="PS_SUPPKEY",
        right_on="S_SUPPKEY"
    )
    .loc[:, ["S_NAME", "S_ADDRESS"]]
    .sort_values("S_NAME")
    .drop_duplicates()
)